# Decoder Benchmark

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
experiment_dir = ROOT / 'experiment/non_exploratory/decoder_benchmarks'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

data_paths = ExperimentPaths(ROOT, experiment_dir / 'config/paths.json')

import json

config = json.loads((experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import sinter
import multiprocessing
from datetime import date
from codes.ca_code import CA_5_2_3, CA_9_2_5, TriangularCode
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders.cpp_backend.bp_osd import BPOSD
from src.decoders.cpp_backend.tesseract import Tesseract
from src.decoders.cpp_backend.adaptive_bp_osd import AdaptiveBPOSD
from src.decoders.cpp_backend.adaptive_bp import AdaptiveBP
from src.decoders.cpp_backend.quotient_split import QuotientSplit
from src.decoders.cpp_backend.mbbp_ld import MBBPLD
from src.decoders.cpp_backend.expanded_adaptive_bp import ExpandedAdaptiveBP

data_dir = data_paths.output('benchmark')
data_dir.mkdir(parents=True, exist_ok=True)
%load_ext autoreload
%autoreload 2


In [ ]:
ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
code_style = {
    'CA_5_2_3': {'color': 'C0', 'label': '$d=3$', 'sort': 0},
    'CA_9_2_5': {'color': 'C1', 'label': '$d=5$', 'sort': 1},
    'CA_14_2_9': {'color': 'C2', 'label': '$d=9$', 'sort': 2},
}

decoder_style = {
    'BPOSD': {
        'marker': '^',
        'linestyle': '-',
        'label': 'BP+OSD',
        'sort': 0,
    },
    'TESSERACT': {
        'marker': 'o',
        'linestyle': '--',
        'label': 'Tesseract',
        'sort': 1,
    },
    'AdaptiveBP': {
        'marker': 's',
        'linestyle': '-.',
        'label': 'Adaptive BP',
        'sort': 2,
    },
    'MBBPLD': {
        'marker': 'v',
        'linestyle': ':',
        'label': 'MBBPLD',
        'sort': 3,
    },
}


In [ ]:
from experiment.utils import benchmark_plotting

group_func = benchmark_plotting.group_func

def plot_args_func(index, curve_id, curve_stats):
    return benchmark_plotting.plot_args_func(index, curve_id, curve_stats, code_style=code_style, decoder_style=decoder_style)


# Benchmark on Different Noise

## 1-round circuit-level noise with perfect measurement

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_05_28_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
from tqdm.auto import tqdm

tasks = []
custom_decoders = {}

for code_name, code in tqdm(ca_codes.items(), desc='Scientific sweep', disable=config['print_level'] == 1):
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0), d_rounds=d_rounds)

        bposd, tess = build_decoder_pair(code, p)

        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=b_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'BPOSD',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=t_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'TESSERACT',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)

if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 1-round circuit-level noise with perfect measurement, no idle

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_15_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_idle_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0, p_idle=0.0), d_rounds=d_rounds)
        bposd, tess = build_decoder_pair(code, p)
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 1-round circuit-level noise with perfect measurement and reset, no idle

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_22_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_reset_idle_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(
            noise_model=noise_model,
            error_rate=ErrorRate(
                p=p,
                p_meas=0.0,
                p_idle=0.0,
                p_prep=0.0,
            ),
            d_rounds=d_rounds,
        )
        bposd, tess = build_decoder_pair(code, p)
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 3-round circuit-level noise

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_22_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
from tqdm.auto import tqdm

tasks = []
custom_decoders = {}

for code_name, code in tqdm(ca_codes.items(), desc='Scientific sweep', disable=config['print_level'] == 1):
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)

        bposd, tess = build_decoder_pair(code, p)

        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=b_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'BPOSD',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=t_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'TESSERACT',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)

if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 1-round phenomenological 

no idle, no reset

In [ ]:
noise_model = 'phenomenological'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_17_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
from tqdm.auto import tqdm

tasks = []
custom_decoders = {}

for code_name, code in tqdm(ca_codes.items(), desc='Scientific sweep', disable=config['print_level'] == 1):
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)

        bposd, tess = build_decoder_pair(code, p)

        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=b_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'BPOSD',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=t_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'TESSERACT',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)

if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 2-round phenomenological 

no idle, no reset

In [ ]:
noise_model = 'phenomenological'
d_rounds = 2
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_17_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
from tqdm.auto import tqdm

tasks = []
custom_decoders = {}

for code_name, code in tqdm(ca_codes.items(), desc='Scientific sweep', disable=config['print_level'] == 1):
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)

        bposd, tess = build_decoder_pair(code, p)

        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=b_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'BPOSD',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=t_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'TESSERACT',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)

if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## code-capacity

In [ ]:
noise_model = 'code_capacity'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_17_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
from tqdm.auto import tqdm

tasks = []
custom_decoders = {}

for code_name, code in tqdm(ca_codes.items(), desc='Scientific sweep', disable=config['print_level'] == 1):
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)

        bposd, tess = build_decoder_pair(code, p)

        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=b_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'BPOSD',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=t_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'TESSERACT',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)

if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


# Benchmark on Different CNOT Schedule

## 1-round circuit-level noise with perfect measurement and no idle, random schedule

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_16_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_idle_random_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code, schedule=1)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0, p_idle=0.0), d_rounds=d_rounds)
        bposd, tess = build_decoder_pair(code, p)
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 1-round circuit-level noise with perfect measurement and reset, no idle, random schedule

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_22_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_reset_idle_random_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code, schedule=1)
    for p in p_values:
        circuit = builder.get_circuit(
            noise_model=noise_model,
            error_rate=ErrorRate(
                p=p,
                p_meas=0.0,
                p_idle=0.0,
                p_prep=0.0,
            ),
            d_rounds=d_rounds,
        )
        bposd, tess = build_decoder_pair(code, p)
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


# Benchmark on Different MP Schedule

## 1-round circuit-level noise with perfect measurement, serial

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_22_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_serial_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0), d_rounds=d_rounds)
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            schedule='serial',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 1-round circuit-level noise with perfect measurement, no idle, serial

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_22_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_idle_serial_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0, p_idle=0.0), d_rounds=d_rounds)
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            schedule='serial',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 1-round circuit-level noise with perfect measurement and no idle, random schedule, serial

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_22_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_idle_random_serial_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code, schedule=1)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0, p_idle=0.0), d_rounds=d_rounds)
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            schedule='serial',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 1-round circuit-level noise with perfect measurement and reset, no idle, serial

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_06_22_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_reset_idle_serial_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(
            noise_model=noise_model,
            error_rate=ErrorRate(
                p=p,
                p_meas=0.0,
                p_idle=0.0,
                p_prep=0.0,
            ),
            d_rounds=d_rounds,
        )
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            schedule='serial',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


# Benchmark on Different alpha

## 1-round circuit-level noise with perfect measurement, alpha=0.8

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
alpha = 0.8
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output(
    '2026_07_23_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_alpha_alpha_shots_decoder_benchmark_max_shots_csv',
    decoder_benchmark_noise_model=noise_model,
    decoder_benchmark_d_rounds=d_rounds,
    alpha=alpha,
    decoder_benchmark_max_shots=max_shots,
)


In [ ]:
from tqdm.auto import tqdm

tasks = []
custom_decoders = {}

for code_name, code in tqdm(ca_codes.items(), desc='Scientific sweep', disable=config['print_level'] == 1):
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0), d_rounds=d_rounds)

        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            ms_scaling_factor=alpha,
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )

        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=b_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'BPOSD',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=t_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'TESSERACT',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)

if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 1-round circuit-level noise with perfect measurement, alpha=1.2

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
alpha = 1.2
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output(
    '2026_07_23_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_perfect_meas_alpha_alpha_shots_decoder_benchmark_max_shots_csv',
    decoder_benchmark_noise_model=noise_model,
    decoder_benchmark_d_rounds=d_rounds,
    alpha=alpha,
    decoder_benchmark_max_shots=max_shots,
)


In [ ]:
from tqdm.auto import tqdm

tasks = []
custom_decoders = {}

for code_name, code in tqdm(ca_codes.items(), desc='Scientific sweep', disable=config['print_level'] == 1):
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0), d_rounds=d_rounds)

        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            ms_scaling_factor=alpha,
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )

        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=b_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'BPOSD',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )
        tasks.append(
            sinter.Task(
                circuit=circuit,
                decoder=t_key,
                json_metadata={
                    'code': code_name,
                    'decoder': 'TESSERACT',
                    'p': float(p),
                    'noise_model': noise_model,
                },
            ),
        )

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)

if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 3-round circuit-level noise, alpha=0.8

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
alpha = 0.8
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output(
    '2026_07_23_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_alpha_alpha_shots_decoder_benchmark_max_shots_csv',
    decoder_benchmark_noise_model=noise_model,
    decoder_benchmark_d_rounds=d_rounds,
    alpha=alpha,
    decoder_benchmark_max_shots=max_shots,
)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            ms_scaling_factor=alpha,
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


# Benchmark on Force OSD

## 1-round circuit-level noise with perfect measurement

In [ ]:
noise_model = 'circuit_level'
d_rounds = 1
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_07_16_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_force_OSD_perfect_meas_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p, p_meas=0.0), d_rounds=d_rounds)
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            force_osd=True,
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 3-round circuit-level noise

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_07_16_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_force_OSD_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            force_osd=True,
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 3-round circuit-level noise: d=9 code

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
codes = {'CA_14_2_9': TriangularCode(3)}
p_values = np.geomspace(0.0001, 0.1, 8)
max_shots = 50000000
max_errors = 1000
save_file = data_paths.output('2026_09_09_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_force_OSD_d_9_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            force_osd=True,
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_14_2_9']['color'], lw=1.5, label=code_style['CA_14_2_9']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


# Benchmark on Adaptive OSD

## 3-round circuit-level noise

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_07_19_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_adaptive_OSD_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        adaptivebposd = AdaptiveBPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            schedule='serial',
        )
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
            schedule='serial',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        ab_key = f'{code_name}_AdaptiveBPOSD_p_{p:.6g}'
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[ab_key] = adaptivebposd
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=ab_key, json_metadata={
            'code': code_name,
            'decoder': 'AdaptiveBPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(16, 10))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=lambda stat: {'label': f"{stat.json_metadata['decoder']} | {stat.json_metadata['code']}"},
    plot_args_func=lambda index,
    curve_id,
    curve_stats: {'marker': 'o'},
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical error rate')
ax.set_ylabel('Logical error rate')
ax.set_title('LER vs p')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()

plt.show()


# Benchmark on Adaptive BP

## 3-round circuit-level noise

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_08_24_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_adaptive_BP_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        adaptivebp = AdaptiveBP(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
        )
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        ab_key = f'{code_name}_AdaptiveBP_p_{p:.6g}'
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[ab_key] = adaptivebp
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=ab_key, json_metadata={
            'code': code_name,
            'decoder': 'AdaptiveBP',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['AdaptiveBP']['marker'],
    linestyle=decoder_style['AdaptiveBP']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['AdaptiveBP']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 3-round circuit-level noise : Compare to MBBP-LD

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
p_values = np.geomspace(0.0001, 0.1, 8)
max_shots = 50000000
max_errors = 1000
codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5(), 'CA_14_2_9': TriangularCode(3)}
save_file = data_paths.output('2026_08_29_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_adaptive_BP_MBBP_LD_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        adaptivebp = AdaptiveBP(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
        )
        mbbpld = MBBPLD(code=code, error_rate=p)
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        ab_key = f'{code_name}_AdaptiveBP_p_{p:.6g}'
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        m_key = f'{code_name}_MBBPLD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[ab_key] = adaptivebp
        custom_decoders[b_key] = bposd
        custom_decoders[m_key] = mbbpld
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=ab_key, json_metadata={
            'code': code_name,
            'decoder': 'AdaptiveBP',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=m_key, json_metadata={
            'code': code_name,
            'decoder': 'MBBPLD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count() // 4,
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
stats = [stat for stat in stats if stat.json_metadata['decoder'] in ['AdaptiveBP', 'BPOSD', 'TESSERACT']]
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_5_2_3']['color'], lw=1.5, label=code_style['CA_5_2_3']['label']), Line2D([0], [0], color=code_style['CA_9_2_5']['color'], lw=1.5, label=code_style['CA_9_2_5']['label']), Line2D([0], [0], color=code_style['CA_14_2_9']['color'], lw=1.5, label=code_style['CA_14_2_9']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['AdaptiveBP']['marker'],
    linestyle=decoder_style['AdaptiveBP']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['AdaptiveBP']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


## 3-round circuit-level noise : Only Adaptive BP on d=9 code

view = 8

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
p_values = np.geomspace(0.0001, 0.1, 8)
max_shots = 50000000
max_errors = 1000
codes = {'CA_14_2_9': TriangularCode(3)}
save_file = data_paths.output('2026_08_30_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_adaptive_BP_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        adaptivebp = AdaptiveBP(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            adaptive_max_views=8,
        )
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        ab_key = f'{code_name}_AdaptiveBP_p_{p:.6g}'
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[ab_key] = adaptivebp
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=ab_key, json_metadata={
            'code': code_name,
            'decoder': 'AdaptiveBP',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count() // 4,
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    highlight_max_likelihood_factor=10,
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
p_vals = [stat.json_metadata['p'] for stat in stats]
ax.set_xlim(min(p_vals) / 1.5, max(p_vals) * 1.5)
ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
ax.tick_params(axis='both', which='major', labelsize=13)
ax.tick_params(axis='both', which='minor', labelsize=0)
distance_handles = [Line2D([0], [0], color=code_style['CA_14_2_9']['color'], lw=1.5, label=code_style['CA_14_2_9']['label'])]
legend1 = ax.legend(
    handles=distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
ax.add_artist(legend1)
decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['BPOSD']['marker'],
    linestyle=decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=decoder_style['TESSERACT']['marker'],
    linestyle=decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=decoder_style['TESSERACT']['label'],
)]
legend2 = ax.legend(
    handles=decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()


# Benchmark on Expanded Adaptive BP

## 3-round circuit-level noise

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
p_values = np.geomspace(0.0001, 0.1, 8)
max_shots = 50000000
max_errors = 1000
codes = {'CA_14_2_9': TriangularCode(3)}
save_file = data_paths.output('2026_09_01_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_expanded_adaptive_BP_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        expanded_adaptive_bp = ExpandedAdaptiveBP(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
        )
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        eab_key = f'{code_name}_ExpandedAdaptiveBP_p_{p:.6g}'
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[eab_key] = expanded_adaptive_bp
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=eab_key, json_metadata={
            'code': code_name,
            'decoder': 'ExpandedAdaptiveBP',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(8, 6))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=lambda stat: {'label': f"{stat.json_metadata['decoder']} | {stat.json_metadata['code']}"},
    plot_args_func=lambda index,
    curve_id,
    curve_stats: {'marker': 'o'},
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical error rate')
ax.set_ylabel('Logical error rate')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()

plt.show()


# Benchmark on Quotient Split BP

## 3-round circuit-level noise

In [ ]:
noise_model = 'circuit_level'
d_rounds = 3
p_values = np.geomspace(1e-08, 0.1, 12)
max_shots = 1000000000
max_errors = 2000
save_file = data_paths.output('2026_08_07_decoder_benchmark_noise_model_decoder_benchmark_d_rounds_quotient_split_shots_decoder_benchmark_max_shots_csv', decoder_benchmark_noise_model=noise_model, decoder_benchmark_d_rounds=d_rounds, decoder_benchmark_max_shots=max_shots)


In [ ]:
tasks = []
custom_decoders = {}
for code_name, code in ca_codes.items():
    builder = CircuitBuilder(code)
    for p in p_values:
        circuit = builder.get_circuit(noise_model=noise_model, error_rate=ErrorRate(p=p), d_rounds=d_rounds)
        quotient_split = QuotientSplit(
            code=code,
            error_rate=p,
            max_iter=50,
            internal_iter=5,
            damping=0.2,
        )
        bposd = BPOSD(
            code=code,
            error_rate=p,
            max_iter=60,
            bp_method='minimum_sum',
            osd_order=10,
            osd_method='osd_cs',
        )
        tess = Tesseract(
            code=code,
            error_rate=p,
            beam=8,
            pq_limit=200000,
            det_penalty=0.0,
        )
        qs_key = f'{code_name}_QuotientSplit_p_{p:.6g}'
        b_key = f'{code_name}_BPOSD_p_{p:.6g}'
        t_key = f'{code_name}_TESSERACT_p_{p:.6g}'
        custom_decoders[qs_key] = quotient_split
        custom_decoders[b_key] = bposd
        custom_decoders[t_key] = tess
        tasks.append(sinter.Task(circuit=circuit, decoder=qs_key, json_metadata={
            'code': code_name,
            'decoder': 'QuotientSplit',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=b_key, json_metadata={
            'code': code_name,
            'decoder': 'BPOSD',
            'p': float(p),
            'noise_model': noise_model,
        }))
        tasks.append(sinter.Task(circuit=circuit, decoder=t_key, json_metadata={
            'code': code_name,
            'decoder': 'TESSERACT',
            'p': float(p),
            'noise_model': noise_model,
        }))

stats = sinter.collect(
    num_workers=multiprocessing.cpu_count(),
    max_shots=max_shots,
    max_errors=max_errors,
    tasks=tasks,
    decoders=list(custom_decoders.keys()),
    custom_decoders=custom_decoders,
    print_progress=config['print_level'] == 3,
    save_resume_filepath=save_file,
)
if config['print_level'] >= 2:
    print(f'Collected stats for {len(stats)} tasks.\n Saved to {save_file}.')


In [ ]:
stats = sinter.stats_from_csv_files(save_file)
fig, ax = plt.subplots(figsize=(16, 10))
sinter.plot_error_rate(
    ax=ax,
    stats=stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=lambda stat: {'label': f"{stat.json_metadata['decoder']} | {stat.json_metadata['code']}"},
    plot_args_func=lambda index,
    curve_id,
    curve_stats: {'marker': 'o'},
)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical error rate')
ax.set_ylabel('Logical error rate')
ax.set_title('LER vs p')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()

plt.show()


# Initial Benchmark

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
initial_benchmark_experiment_dir = ROOT / 'experiment/non_exploratory/decoder_benchmarks'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

initial_benchmark_data_paths = ExperimentPaths(ROOT, initial_benchmark_experiment_dir / 'config/paths.json')

import json

initial_benchmark_config = json.loads((initial_benchmark_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations

initial_benchmark_data_dir = initial_benchmark_data_paths.run()
initial_benchmark_data_dir.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD as initial_benchmark_bposd
from src.decoders import Tesseract as initial_benchmark_Tesseract


In [ ]:
initial_benchmark_ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def initial_benchmark_build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=initial_benchmark_bposd, tesseract_type=initial_benchmark_Tesseract)


# Phenomenon Observation

See the discrepancy between BPOSD and Tesseract in the benchmark

In [ ]:
from experiment.utils import benchmark_plotting

stats_to_df = benchmark_plotting.stats_to_df
plot_gap_interval = benchmark_plotting.plot_gap_interval


## 1-round circuit-level noise with perfect measurement

In [ ]:
initial_benchmark_noise_model = 'circuit_level'
initial_benchmark_d_rounds = 1
initial_benchmark_p_values = np.geomspace(1e-08, 0.1, 12)
initial_benchmark_max_shots = 1000000000
initial_benchmark_max_errors = 2000
initial_benchmark_save_file = initial_benchmark_data_paths.output('2026_05_28_initial_benchmark_noise_model_initial_benchmark_d_rounds_perfect_meas_shots_initial_benchmark_max_shots_csv', initial_benchmark_noise_model=initial_benchmark_noise_model, initial_benchmark_d_rounds=initial_benchmark_d_rounds, initial_benchmark_max_shots=initial_benchmark_max_shots)


In [ ]:
assert initial_benchmark_save_file.is_file(), 'Run the main baseline collection section first'
initial_benchmark_stats = sinter.stats_from_csv_files(initial_benchmark_save_file)


In [ ]:
initial_benchmark_stats = sinter.stats_from_csv_files(initial_benchmark_save_file)
df = stats_to_df(initial_benchmark_stats)
summary = df.pivot_table(
    index=['code', 'p'],
    columns='decoder',
    values='logical_error_rate',
    aggfunc='mean',
).reset_index().rename_axis(None, axis=1)
summary['gap_bposd_minus_tess'] = summary['BPOSD'] - summary['TESSERACT']
summary = summary.sort_values(['code', 'p']).reset_index(drop=True)


In [ ]:
initial_benchmark_fig, initial_benchmark_ax = plt.subplots(figsize=(8, 5))
sinter.plot_error_rate(
    ax=initial_benchmark_ax,
    stats=initial_benchmark_stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=lambda stat: {'label': f"{stat.json_metadata['decoder']} | {stat.json_metadata['code']}"},
    plot_args_func=lambda index,
    curve_id,
    curve_stats: {'marker': 'o'},
)
initial_benchmark_ax.set_xscale('log')
initial_benchmark_ax.set_yscale('log')
initial_benchmark_ax.set_xlabel('Physical error rate')
initial_benchmark_ax.set_ylabel('Logical error rate')
initial_benchmark_ax.set_title('LER vs p')
initial_benchmark_ax.grid(True, which='both', alpha=0.3)
initial_benchmark_ax.legend()
plt.tight_layout()

plt.show()


In [ ]:
plot_gap_interval(summary, x_scale=('log', 'log'), y_scale=('log', 'log'))


# Benchmark Figures
Configuration: `config`. Figures remain local; datasets are under `experiment/data/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
benchmark_figures_experiment_dir = ROOT / 'experiment/non_exploratory/decoder_benchmarks'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

benchmark_figures_data_paths = ExperimentPaths(ROOT, benchmark_figures_experiment_dir / 'config/paths.json')

import json

benchmark_figures_config = json.loads((benchmark_figures_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import sinter

benchmark_figures_save_file = str(benchmark_figures_data_paths.output('2026_06_22_circuit_level_3_shots_1000000000_csv'))


In [ ]:
benchmark_figures_stats = sinter.stats_from_csv_files(benchmark_figures_save_file)
benchmark_figures_fig, benchmark_figures_ax = plt.subplots(figsize=(8, 6))
sinter.plot_error_rate(
    ax=benchmark_figures_ax,
    stats=benchmark_figures_stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=lambda stat: {'label': f"{stat.json_metadata['decoder']} | {stat.json_metadata['code']}"},
    plot_args_func=lambda index,
    curve_id,
    curve_stats: {'marker': 'o'},
)
benchmark_figures_ax.set_xscale('log')
benchmark_figures_ax.set_yscale('log')
benchmark_figures_ax.set_xlabel('Physical error rate')
benchmark_figures_ax.set_ylabel('Logical error rate')
benchmark_figures_ax.grid(True, which='both', alpha=0.3)
benchmark_figures_ax.legend()
plt.tight_layout()

plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import sinter

benchmark_figures_stats = sinter.stats_from_csv_files(benchmark_figures_save_file)
benchmark_figures_code_style = {
    'CA_5_2_3': {'color': 'C0', 'label': '$d=3$', 'sort': 0},
    'CA_9_2_5': {'color': 'C1', 'label': '$d=5$', 'sort': 1},
    'CA_14_2_9': {'color': 'C2', 'label': '$d=9$', 'sort': 2},
}

benchmark_figures_decoder_style = {
    'BPOSD': {
        'marker': '^',
        'linestyle': '-',
        'label': 'BP+OSD',
        'sort': 0,
    },
    'TESSERACT': {
        'marker': 'o',
        'linestyle': '--',
        'label': 'Tesseract',
        'sort': 1,
    },
}

from experiment.utils import benchmark_plotting

benchmark_figures_group_func = benchmark_plotting.group_func

def benchmark_figures_plot_args_func(index, curve_id, curve_stats):
    return benchmark_plotting.plot_args_func(index, curve_id, curve_stats, code_style=benchmark_figures_code_style, decoder_style=benchmark_figures_decoder_style)

benchmark_figures_fig, benchmark_figures_ax = plt.subplots(figsize=(7.2, 5.6))
sinter.plot_error_rate(
    ax=benchmark_figures_ax,
    stats=benchmark_figures_stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=benchmark_figures_group_func,
    plot_args_func=benchmark_figures_plot_args_func,
    highlight_max_likelihood_factor=10,
)
benchmark_figures_ax.set_xscale('log')
benchmark_figures_ax.set_yscale('log')
benchmark_figures_ax.set_xlabel('Physical Error Rate, $p$', fontsize=18)
benchmark_figures_ax.set_ylabel('Logical Error Rate, $\\hat{P}_L$', fontsize=18)
benchmark_figures_p_vals = [stat.json_metadata['p'] for stat in benchmark_figures_stats]
benchmark_figures_ax.set_xlim(min(benchmark_figures_p_vals) / 1.5, max(benchmark_figures_p_vals) * 1.5)
benchmark_figures_ax.grid(True, which='major', alpha=0.28, linewidth=0.8)
benchmark_figures_ax.grid(True, which='minor', alpha=0.1, linewidth=0.5)
benchmark_figures_ax.tick_params(axis='both', which='major', labelsize=13)
benchmark_figures_ax.tick_params(axis='both', which='minor', labelsize=0)
benchmark_figures_distance_handles = [Line2D([0], [0], color=benchmark_figures_code_style['CA_5_2_3']['color'], lw=1.5, label=benchmark_figures_code_style['CA_5_2_3']['label']), Line2D([0], [0], color=benchmark_figures_code_style['CA_9_2_5']['color'], lw=1.5, label=benchmark_figures_code_style['CA_9_2_5']['label']), Line2D([0], [0], color=benchmark_figures_code_style['CA_14_2_9']['color'], lw=1.5, label=benchmark_figures_code_style['CA_14_2_9']['label'])]
benchmark_figures_legend1 = benchmark_figures_ax.legend(
    handles=benchmark_figures_distance_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.34),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
benchmark_figures_ax.add_artist(benchmark_figures_legend1)
benchmark_figures_decoder_handles = [Line2D(
    [0],
    [0],
    color='black',
    marker=benchmark_figures_decoder_style['BPOSD']['marker'],
    linestyle=benchmark_figures_decoder_style['BPOSD']['linestyle'],
    lw=1.2,
    markersize=6,
    label=benchmark_figures_decoder_style['BPOSD']['label'],
), Line2D(
    [0],
    [0],
    color='black',
    marker=benchmark_figures_decoder_style['TESSERACT']['marker'],
    linestyle=benchmark_figures_decoder_style['TESSERACT']['linestyle'],
    lw=1.2,
    markersize=5,
    label=benchmark_figures_decoder_style['TESSERACT']['label'],
)]
benchmark_figures_legend2 = benchmark_figures_ax.legend(
    handles=benchmark_figures_decoder_handles,
    loc='lower right',
    bbox_to_anchor=(0.98, 0.08),
    frameon=False,
    fontsize=13,
    handlelength=1.8,
)
plt.tight_layout()

plt.show()
